# G2 / D10 — E5 joint trained screening (TV1 + TV2)

Notebook orchestration dùng `scripts/run_g2.py --stage joint_search`; không có training loop/model/metric riêng.
TV1: `can,fabric,fruit_jelly,rice` (180 ô). TV2: `sheet_metal,vial,wallplugs,walnuts` (180 ô).
9 cặp r/d × 5 Fusion × 8 category = 360 ô screening E5. Một model mới cho mỗi category/r/d/Fusion.
Không train main E1–E3, không tự chọn cấu hình hoặc tạo selection lock.

Mặc định do người dùng chốt: **20 epochs, batch 32, AdamW**; ngân sách main được ghi riêng.
T4/L4/A100 phải xác nhận bằng smoke thật; CPU PASS không chứng minh batch 32/64 vừa VRAM.
Drive chỉ sync sau epoch và khi job xong/lỗi. Mở notebook không tự chạy full search.

**Trước khi chạy:** push/merge code D10 rồi đặt `GIT_COMMIT` đúng SHA đó. Hai thành viên dùng cùng SHA,
DINO commit/checkpoint, archives và config. Thay budget/batch/seed tạo protocol namespace mới;
không hạ batch riêng cho một candidate sau khi xem DEV.


## 1. Setup — Input, output riêng, GPU, Git, dependencies, Drive

In [ ]:
# @title INPUT / Git — chỉnh đúng cấu trúc Drive của bạn
from pathlib import Path
import os, sys, json, subprocess, shutil

GITHUB_URL = 'https://github.com/haduckien-raccoon/msila.git'
GITHUB_REF = 'g2/member1'  # Sau merge có thể dùng g2/integration.
GIT_COMMIT = ''  # SHA chứa D10; resume và TV2 dùng cùng SHA.
DINO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINO_COMMIT = 'main'  # Resolve một lần; dùng SHA manifest cũ khi resume.
DRIVE_PROJECT = Path('/content/drive/MyDrive/msila')
ARCHIVE_DIR = DRIVE_PROJECT / 'data'
CATEGORIES = ['can','fabric','fruit_jelly','rice','sheet_metal','vial','wallplugs','walnuts']
ARCHIVE_NAMES = {category: f'{category}.tar.gz' for category in CATEGORIES}
CHECKPOINT_ON_DRIVE = DRIVE_PROJECT / 'models/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'


In [ ]:
# @title OUTPUT — tất cả đường dẫn output tại riêng cell này
DRIVE_OUTPUT_BASE = DRIVE_PROJECT / 'outputs/G2/D10'
LOCAL_OUTPUT_BASE = Path('/content/msila_d10/outputs')
RUN_ID = 'e5_screen_e20_b32_v1'
WORK = Path('/content/msila_d10')
REPO = Path('/content/msila')
DINO = Path('/content/dinov3')
LOCAL_ARCHIVES = WORK / 'archives'
DATA = Path('/content/datasets/mvtec_ad2')
LOCAL_WEIGHTS = WORK / 'weights' / CHECKPOINT_ON_DRIVE.name
VENV = WORK / 'venv'
MODEL_CONFIG = WORK / 'config/model.yaml'
BASE_RUNNER_CONFIG = WORK / 'config/base_runner.yaml'
JOINT_CONFIG = WORK / 'config/joint_search.yaml'


In [ ]:
# @title SCREENING — quyết định chung trước khi chạy bất kỳ candidate
OWNER = 'tv1'  # TV2 chỉ đổi thành 'tv2'.
SCREENING_EPOCHS = 20
SCREENING_BATCH_SIZE = 32  # Có thể đặt 64 TRƯỚC smoke/search với protocol mới.
SCREENING_MAX_STEPS = None
DATA_WORKERS = 2
PAIRED_STRATEGY = 'sequential'  # Cùng backbone, hai forward B thay vì một 2B.
SELECTED_CATEGORIES = []  # [] = đủ 4 category của OWNER; hoặc ['can'].
PAIR_FILTER = None       # None = đủ 9; hoặc (64,256).
FUSION_FILTER = []        # [] = đủ 5; hoặc ['mean','attention'].
MAX_JOBS_THIS_SESSION = 1  # Đổi None để chạy cả phần còn lại; manifest vẫn đủ 180.
RESUME = True
RUN_GPU_SMOKE = True  # Một job, tối đa 2 updates; không tính screening PASS.
RUN_FULL_SEARCH = False  # Chỉ bật ở cell full search khi muốn tiêu ngân sách GPU.
SMOKE_PAIR = (256,768)
SMOKE_FUSION = 'attention'
assert OWNER in ('tv1','tv2')
assert type(SCREENING_EPOCHS) is int and SCREENING_EPOCHS > 0
assert type(SCREENING_BATCH_SIZE) is int and SCREENING_BATCH_SIZE > 0


In [ ]:
# @title Mount Drive và checkout commit
from google.colab import drive
drive.mount('/content/drive')
for directory in (WORK, LOCAL_OUTPUT_BASE, DRIVE_OUTPUT_BASE, WORK/'config', WORK/'tmp', WORK/'mpl'):
    directory.mkdir(parents=True, exist_ok=True)

def checkout_pinned(url, directory, ref):
    if not (directory/'.git').exists():
        subprocess.run(['git','clone','--depth','1',url,str(directory)], check=True)
    dirty = subprocess.check_output(['git','-C',str(directory),'status','--porcelain','--untracked-files=no'], text=True)
    if dirty.strip():
        raise RuntimeError(f'Checkout có tracked changes chưa lưu: {directory}')
    subprocess.run(['git','-C',str(directory),'fetch','--depth','1','origin',ref], check=True)
    commit = subprocess.check_output(['git','-C',str(directory),'rev-parse','FETCH_HEAD'], text=True).strip()
    subprocess.run(['git','-C',str(directory),'checkout','--detach',commit], check=True)
    return commit

MSILA_COMMIT = checkout_pinned(GITHUB_URL, REPO, GIT_COMMIT or GITHUB_REF)
if not (REPO/'src/train/g2_joint_search.py').is_file():
    raise RuntimeError('Commit chưa có D10. Push/merge D10 rồi đặt GIT_COMMIT theo SHA mới.')
DINO_SHA = checkout_pinned(DINO_URL, DINO, DINO_COMMIT)
print('MS-ILA:', MSILA_COMMIT, 'DINOv3:', DINO_SHA)

if not (VENV/'bin/python').is_file():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
requirements = [line for line in (REPO/'requirements.txt').read_text().splitlines()
                if line.strip() and not line.lstrip().startswith(('#','torch','safetensors'))]
requirements.append('safetensors>=0.8')
dependency_file = WORK/'requirements_colab.txt'
dependency_file.write_text('\n'.join(requirements)+'\n')
subprocess.run([PYTHON,'-m','pip','install','-q','-r',str(dependency_file)], check=True)
RUN_ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
               CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLCONFIGDIR=str(WORK/'mpl'), TMPDIR=str(WORK/'tmp'))
subprocess.run([PYTHON,'-c',"import torch,safetensors; from packaging.version import Version; assert Version(safetensors.__version__) >= Version('0.8'); assert torch.cuda.is_available(), 'GPU NOT RUN'; print(torch.cuda.get_device_name(0), torch.__version__, safetensors.__version__)"], env=RUN_ENV, check=True)


## 2. Preflight — copy archives → extract trên Colab → kiểm tra config/weights

Helper dataset hiện có chỉ lấy `TRAIN/good` và `VALIDATION/good`, không sử dụng TEST.
Copy đủ tám `.tar.gz` vào ổ Colab trước khi mở bất kỳ archive nào. DINO checkpoint cũng copy local.
Preflight kiểm tra checksum, split, registry ViT-B/16 768/block 4/8/12 và đủ 45 D9 PASS;
frozen model/gradient được kiểm chứng bằng smoke ở phần tiếp theo.


In [ ]:
# @title Chuẩn bị dữ liệu, weights, config và resume Drive
prepare_code = """
import json,sys
from pathlib import Path
from scripts.eval_g2 import prepare_archives,copy_atomic,sync_tree
from src.train.screen_representation import read_yaml,save_yaml,save_json,sha256_file
p=json.loads(sys.argv[1])
prepare_archives(p['archive_dir'],p['archives'],p['local_archives'],p['data'])
source=Path(p['checkpoint']); target=Path(p['weights'])
if not source.is_file(): raise FileNotFoundError(source)
checksum=sha256_file(source)
if not target.is_file() or sha256_file(target)!=checksum: copy_atomic(source,target)
assert sha256_file(target)==checksum
cfg=read_yaml(Path('configs/g2_experiments.yaml'))
cfg['backbone'].update(name='dinov3_vitb16',repo_dir=p['dino'],weights=p['weights'])
save_yaml(cfg,Path(p['model_config']))
base=read_yaml(Path('configs/g2_runner.yaml')); base['model_config']=p['model_config']
save_yaml(base,Path(p['base_config']))
setting=read_yaml(Path('configs/g2_joint_search.yaml'))
setting.update(base_config=p['base_config'],run_id=p['run_id'],paired_strategy=p['pair_strategy'],num_workers=p['workers'])
setting['screening_budget'].update(epochs=p['epochs'],batch_size=p['batch'],max_steps=p['max_steps'])
save_yaml(setting,Path(p['joint_config']))
sync_tree(Path(p['drive_output'])/p['run_id'],Path(p['output'])/p['run_id'])
environment=dict(msila_commit=p['msila_commit'],dino_commit=p['dino_commit'],checkpoint_sha256=checksum,
                 screening_budget=setting['screening_budget'],drive_sync='epoch_and_job_end')
save_json(environment,Path(p['output'])/p['run_id']/'notebook_environment.json')
copy_atomic(Path(p['output'])/p['run_id']/'notebook_environment.json',Path(p['drive_output'])/p['run_id']/'notebook_environment.json')
print(json.dumps(environment,indent=2))
"""
payload = dict(archive_dir=str(ARCHIVE_DIR), archives=ARCHIVE_NAMES, local_archives=str(LOCAL_ARCHIVES),
               data=str(DATA), checkpoint=str(CHECKPOINT_ON_DRIVE), weights=str(LOCAL_WEIGHTS), dino=str(DINO),
               model_config=str(MODEL_CONFIG), base_config=str(BASE_RUNNER_CONFIG), joint_config=str(JOINT_CONFIG),
               run_id=RUN_ID, output=str(LOCAL_OUTPUT_BASE), drive_output=str(DRIVE_OUTPUT_BASE),
               epochs=SCREENING_EPOCHS,batch=SCREENING_BATCH_SIZE,max_steps=SCREENING_MAX_STEPS,
               pair_strategy=PAIRED_STRATEGY,workers=DATA_WORKERS,msila_commit=MSILA_COMMIT,dino_commit=DINO_SHA)
subprocess.run([PYTHON,'-c',prepare_code,json.dumps(payload)], cwd=REPO, env=RUN_ENV, check=True)

COMMON = ['--stage','joint_search','--config',str(JOINT_CONFIG),'--owner',OWNER,'--device','cuda',
          '--output-root',str(LOCAL_OUTPUT_BASE),'--drive-output',str(DRIVE_OUTPUT_BASE)]

def run_joint(arguments, label, allow_incomplete=False):
    # Streaming console log stays LOCAL; runner syncs only at epoch/job boundaries.
    log_path = LOCAL_OUTPUT_BASE / RUN_ID / f'colab_{OWNER}_{label}.log'
    log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('a') as log:
        process=subprocess.Popen([PYTHON,'scripts/run_g2.py',*COMMON,*arguments],cwd=REPO,env=RUN_ENV,
                                 stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end=''); log.write(line); log.flush()
        result=process.wait()
    destination=DRIVE_OUTPUT_BASE / RUN_ID / log_path.name
    shutil.copy2(log_path,destination)
    if result != 0 and not (allow_incomplete and result == 2):
        raise RuntimeError(f'{label} exit={result}; xem logs và FAIL/OOM manifest trên Drive')
    return result

run_joint(['--prepare'],'plan')
run_joint(['--preflight'],'preflight')


## 3. Smoke — một E5 job thật, không phải screening evidence

Hai updates tối đa, một epoch với source subset; kiểm tra forward/backward, loss, frozen backbone,
metric native và checkpoint/save. Mặc định chọn cặp lớn nhất để phát hiện vấn đề VRAM sớm.
Nếu OOM, đổi batch chung ở cell SCREENING trước khi tạo protocol mới; không tự giảm riêng từng Fusion.


In [ ]:
# @title GPU smoke 1 job
SMOKE_OK = False
if RUN_GPU_SMOKE:
    smoke_category = 'can' if OWNER == 'tv1' else 'sheet_metal'
    smoke_args=['--smoke','--categories',smoke_category,'--r',str(SMOKE_PAIR[0]),'--d',str(SMOKE_PAIR[1]),
                '--fusions',SMOKE_FUSION,'--max-jobs','1']
    if RESUME: smoke_args.append('--resume')
    run_joint(smoke_args,'smoke')
    SMOKE_OK = True
else:
    print('GPU smoke: NOT RUN. Full cell yêu cầu smoke PASS trong phiên này.')


## 4. Full screening — run/resume/skip theo nhóm

Cell riêng, mặc định tắt. Dùng `SELECTED_CATEGORIES`, `PAIR_FILTER`, `FUSION_FILTER`, `MAX_JOBS_THIS_SESSION`
để chia công việc; manifest luôn giữ đủ 180 ô của owner và skeleton chung 360 ô.
Skip chỉ khi config/candidate/seed/budget/checkpoint/DEV và evaluator artifacts được audit hợp lệ.
Drive lưu local checkpoints sau từng epoch, thêm config/log/metrics sau từng job. Không chạy main training.


In [ ]:
# @title FULL — bật RUN_FULL_SEARCH ở cell SCREENING khi sẵn sàng
if RUN_FULL_SEARCH:
    assert SMOKE_OK, 'Chạy smoke GPU PASS trong phiên này trước full screening'
    full_args=[]
    if SELECTED_CATEGORIES: full_args += ['--categories',*SELECTED_CATEGORIES]
    if PAIR_FILTER: full_args += ['--r',str(PAIR_FILTER[0]),'--d',str(PAIR_FILTER[1])]
    if FUSION_FILTER: full_args += ['--fusions',*FUSION_FILTER]
    if MAX_JOBS_THIS_SESSION is not None: full_args += ['--max-jobs',str(MAX_JOBS_THIS_SESSION)]
    if RESUME: full_args.append('--resume')
    run_joint(full_args,'screening',allow_incomplete=True)
else:
    print('Full E5 screening: NOT RUN — RUN_FULL_SEARCH=False')


## 5. Coverage, handoff và output

`PASS/N/180` chỉ đếm training đủ budget + pretrained thật + GPU + evaluator artifacts được kiểm chứng.
Smoke và CPU fixture không được tính. Hai TV dùng cùng notebook, commit/config/weights/splits;
TV2 đổi OWNER='tv2' để chạy 180 ô còn lại. Mỗi owner có folder/manifest riêng để không ghi đè khi chạy song song.
Gửi TV2 `notebook_environment.json`, shared config, `training_protocol.json`, spec và commit SHA.
Sau đó TV2 ghép evidence 360 ô và dùng evaluator/summarizer hiện có; TV1 không tạo selection lock.


In [ ]:
# @title Audit coverage đã có và chỉ dẫn handoff
run_joint(['--report-only'],'coverage',allow_incomplete=True)
for path in sorted((LOCAL_OUTPUT_BASE/RUN_ID).glob(f'*/{OWNER}/training_coverage.json')):
    report=json.loads(path.read_text())
    print(path.parent, f"PASS {report['real_pass']}/{report['expected']}",report['status'])
    print(json.dumps(report,indent=2,ensure_ascii=False))
print('Drive:', DRIVE_OUTPUT_BASE/RUN_ID)
print('TV2 dùng cùng MSILA_COMMIT:', MSILA_COMMIT, 'DINO_SHA:', DINO_SHA, "và OWNER='tv2'")
print('Artifacts: *_joint_search_180.csv, joint_search_360.csv, joint_search_spec.json, training_protocol.json,')
print('training_coverage.json, train/**/best.pt + last.pt + SHA + config + loss_history.json + train.log + metrics.json,')
print('evaluation/**/native_maps.json + metrics.json + QA + native score/mask exports.')
print('Chưa có đủ kết quả 8 category thì không macro/selection lock và không báo khoa học PASS.')
